In [1]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [2]:
import kagglehub
import shutil

# Download latest version
path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
destination = "/content/crop-yield-prediction-dataset"
shutil.copytree(path, destination, dirs_exist_ok=True)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
Path to dataset files: /kaggle/input/crop-recommendation-dataset


In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv("/content/crop-yield-prediction-dataset/Crop_recommendation.csv")
df = pd.DataFrame(data)
df.head()


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# represents all features except the pred lab
X = df.drop('label', axis=1).to_numpy()
y = df['label'].to_numpy()

encoder = LabelEncoder()
y = encoder.fit_transform(y)

Xtr, Xte, ytr, yte = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

Xtr = scaler.fit_transform(Xtr)
Xte = scaler.transform(Xte)

In [5]:
import torch

dev = "cuda" if torch.cuda.is_available() else "cpu"

Xtr = torch.tensor(Xtr, dtype=torch.float32)
Xte = torch.tensor(Xte, dtype=torch.float32)

ytr = torch.tensor(ytr, dtype=torch.long)
yte = torch.tensor(yte, dtype=torch.long)

print("Device:", dev)

Device: cuda


In [6]:
from torch.utils.data import Dataset, DataLoader

class FarmDS(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

In [7]:
train_ds = FarmDS(Xtr, ytr)
test_ds = FarmDS(Xte, yte)

print("Training samples:", len(train_ds))
print("Testing samples:", len(test_ds))

print(train_ds[0])

Training samples: 1760
Testing samples: 440
(tensor([-1.3716, -1.0729, -0.6735,  0.8195,  0.9244,  0.9375,  0.1994]), tensor(16))


In [8]:
train_dl = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True
)

test_dl = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False
)

In [9]:
for i, (xb, yb) in enumerate(train_dl):
    print(
        f"Batch {i+1} | "
        f"Features: {xb.shape} | "
        f"Labels: {yb.shape}"
    )

Batch 1 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 2 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 3 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 4 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 5 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 6 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 7 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 8 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 9 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 10 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 11 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 12 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 13 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 14 | Features: torch.Size([32, 7]) | Labels: torch.Size([32])
Batch 15 | Features: torch.Size([32, 7]) | Labels: torch.

In [10]:
print("Number of training batches:", len(train_dl))
print("Number of testing batches:", len(test_dl))

xb, yb = next(iter(train_dl))

print("Batch feature shape:", xb.shape)
print("Batch label shape:", yb.shape)
print("First 5 labels:", yb[:5])

Number of training batches: 55
Number of testing batches: 14
Batch feature shape: torch.Size([32, 7])
Batch label shape: torch.Size([32])
First 5 labels: tensor([16, 13, 14, 11,  5])
